# Taller 2: ETL con Apache Spark y Delta Lake en Databricks

En este taller construirás un flujo ETL completo sobre datos de ventas aplicando la **arquitectura Medallion** (Bronze → Silver → Gold) con **Apache Spark** y **Delta Lake** como formato de tabla abierto.

## Objetivos de aprendizaje
- Configurar Databricks Community Edition desde cero
- Implementar la arquitectura Lakehouse Medallion
- Leer, transformar y escribir datos con la API de DataFrames de PySpark
- Usar Delta Lake: transacciones ACID, versionado y time travel
- Ejecutar consultas analíticas sobre tablas Gold

## Contenido
0. Configuración del entorno (Databricks CE)
1. Bronze — Ingesta de datos crudos
2. Silver — Limpieza y transformación
3. Gold — Métricas y agregaciones
4. Delta Lake — Capacidades avanzadas
5. Ejercicios integradores

---
## 0. Configuración del entorno

Sigue estos pasos **una sola vez** antes de ejecutar el notebook.

---

### Paso 1 — Crear una cuenta en Databricks Community Edition

Databricks ofrece una capa gratuita llamada **Community Edition** que no requiere tarjeta de crédito.

1. Ve a https://www.databricks.com/try-databricks
2. Haz clic en **"Get started for free"** o **"Try Databricks"**
3. Completa el formulario de registro (nombre, correo, empresa/universidad)
4. En la pantalla de selección de plan, elige **"Community Edition"** (opción gratuita)
5. Verifica tu correo electrónico y accede a https://community.cloud.databricks.com

> **Limitaciones de Community Edition:** un solo cluster de 1 nodo, se apaga automáticamente tras 2 horas de inactividad. Es suficiente para este taller.

---

### Paso 2 — Crear un cluster

El cluster es el motor de cómputo donde correrá Spark.

1. En la barra lateral izquierda, haz clic en **Compute**
2. Haz clic en **"Create compute"** (o **"+ New cluster"**)
3. Completa los campos:
   - **Cluster name:** `taller2-cluster` (o cualquier nombre)
   - **Databricks Runtime:** elige la versión más reciente con etiqueta **LTS** (ej. `14.x LTS` o superior)
   - Las demás opciones se configuran solas en Community Edition
4. Haz clic en **"Create compute"**
5. Espera 3-5 minutos hasta que el estado cambie a **Running** (círculo verde)

> El runtime LTS incluye Apache Spark y Delta Lake preinstalados. No necesitas instalar nada adicional.

---

### Paso 3 — Importar este notebook en Databricks

**Opción A — Importar desde archivo .ipynb:**
1. En la barra lateral, ve a **Workspace** → tu carpeta de usuario (ícono de persona)
2. Haz clic derecho → **Import**
3. Selecciona **"File"**, arrastra o selecciona el archivo `taller_databricks_etl.ipynb`
4. Haz clic en **Import**

**Opción B — Crear notebook nuevo y pegar el código:**
1. Ve a **Workspace** → **Create** → **Notebook**
2. Nómbralo `taller_databricks_etl`, elige Python como lenguaje
3. Copia y pega cada celda de código manualmente

---

### Paso 4 — Conectar el notebook al cluster

1. Abre el notebook en Databricks
2. En la parte superior derecha verás un menú desplegable que dice **"Detached"** o **"Connect"**
3. Haz clic → selecciona el cluster `taller2-cluster` que creaste
4. El estado cambia a **"Attached"** o muestra el nombre del cluster

> En Databricks, `spark` y `dbutils` están disponibles automáticamente. No necesitas importar ni inicializar `SparkSession`.

---

### Paso 5 — Ejecutar las celdas

- Ejecutar celda actual: `Shift + Enter`
- Ejecutar todo el notebook: menú **Run** → **Run all**
- Ejecutar desde la celda actual: menú **Run** → **Run all below**

Ejecuta las celdas **en orden de arriba hacia abajo** la primera vez.

---
## Arquitectura del taller

```
  FUENTE DE DATOS
  (datos sintéticos)
        |
        v
 +--------------+
 |    BRONZE    |  Datos crudos tal como llegan
 |  Delta Lake  |  Sin transformaciones
 +--------------+
        |
        v
 +--------------+
 |    SILVER    |  Datos limpios y enriquecidos
 |  Delta Lake  |  Nulos resueltos, tipos correctos,
 +--------------+  columnas derivadas
        |
        v
 +--------------+
 |     GOLD     |  Métricas y agregaciones
 |  Delta Lake  |  Listas para consumo analítico
 +--------------+
```

**Formato de almacenamiento:** Delta Lake (formato de tabla abierto basado en Parquet + transaction log)

**Ubicación:** DBFS (Databricks File System) en la ruta `/tmp/taller2/`

In [0]:
# =============================================================
# CONFIGURACIÓN — Compatible con Serverless + Unity Catalog
# DBFS root está deshabilitado en el nuevo CE.
# Usamos tablas Delta administradas en el catálogo por defecto.
# =============================================================
from pyspark.sql import functions as F

# Nombre de la base de datos (schema) que usaremos
DB_NAME = "taller2"

# Crear el schema si no existe (usa el catálogo por defecto: 'main')
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB_NAME}")
spark.sql(f"USE {DB_NAME}")

# Limpiar ejecuciones anteriores
for tabla in ["bronze_ventas", "silver_ventas", 
              "gold_cat_mes", "gold_top_clientes", 
              "gold_metodos_pago", "gold_tendencia_semanal"]:
    spark.sql(f"DROP TABLE IF EXISTS {DB_NAME}.{tabla}")

print(f"Schema '{DB_NAME}' listo.")
print(f"Catálogo activo : {spark.catalog.currentCatalog()}")
print(f"Schema activo   : {spark.catalog.currentDatabase()}")
print(f"Spark version   : {spark.version}")

Schema 'taller2' listo.
Catálogo activo : workspace
Schema activo   : taller2
Spark version   : 4.1.0


---
## Parte 1 — Bronze: Ingesta de datos crudos

La capa Bronze almacena los datos **exactamente como llegan** desde la fuente, sin ninguna transformación.  
Su propósito es preservar el dato original para auditoría y reprocesamiento.

En este taller generaremos datos sintéticos de transacciones de ventas que simulan la llegada de un archivo CSV externo.  
Los datos incluyen **problemas de calidad intencionales** que corregiremos en Silver:
- ~10% de valores nulos en `discount_pct`
- ~3% de precios inválidos (valores negativos)

In [0]:
# =============================================================
# 1.1 — GENERAR DATOS SINTÉTICOS DE VENTAS
# Usamos funciones nativas de Spark para generar 1.000 filas.
# Esto simula la llegada de un archivo externo (CSV, JSON, etc.)
# =============================================================

from pyspark.sql import functions as F

SEED = 42  # semilla para reproducibilidad

# Listas de valores posibles
arr_categorias = F.array([F.lit(c) for c in ["Electronica", "Muebles", "Ropa", "Alimentos", "Libros"]])
arr_ciudades   = F.array([F.lit(c) for c in ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena"]])
arr_pagos      = F.array([F.lit(p) for p in ["tarjeta_credito", "tarjeta_debito", "efectivo", "transferencia"]])
arr_productos  = F.array([F.lit(p) for p in [
    "Laptop Pro", "Silla Ergonomica", "Camiseta Algodon", "Arroz Premium",
    "Python Avanzado", "Monitor 4K", "Escritorio Pie", "Jean Clasico",
    "Aceite Oliva", "SQL para Todos", "Auriculares BT", "Estante Madera",
    "Zapatos Cuero", "Leche Entera", "Historia IA"
]])

# Generamos 1.000 filas con spark.range y columnas aleatorias
df_bronze_raw = (
    spark.range(1, 1001)
    .select(
        F.col("id"),
        F.rand(SEED).alias("r1"),
        F.rand(SEED + 1).alias("r2"),
        F.rand(SEED + 2).alias("r3"),
        F.rand(SEED + 3).alias("r4"),
        F.rand(SEED + 4).alias("r5"),
        F.rand(SEED + 5).alias("r6"),
        F.rand(SEED + 6).alias("r7"),
    )
    .select(
        # ID de transacción: TXN-00001 ... TXN-01000
        F.concat(F.lit("TXN-"), F.lpad(F.col("id").cast("string"), 5, "0")).alias("transaction_id"),
        # ID cliente: C0001 ... C0200
        F.concat(F.lit("C"), F.lpad(((F.col("r1") * 200).cast("int") + 1).cast("string"), 4, "0")).alias("customer_id"),
        # Producto y categoría (de las listas)
        arr_productos[(F.col("r2") * 15).cast("int")].alias("product_name"),
        arr_categorias[(F.col("r3") * 5).cast("int")].alias("category"),
        # Cantidad entre 1 y 10
        ((F.col("r4") * 9).cast("int") + 1).alias("quantity"),
        # Precio: ~3% de valores -99 (inválidos) para limpiar en Silver
        F.when(F.col("r5") < 0.03, F.lit(-99.0))
         .otherwise(F.round(F.col("r5") * 1995 + 5, 2)).alias("unit_price"),
        # Fecha entre 2024-01-01 y 2024-12-31
        F.date_add(F.lit("2024-01-01").cast("date"), (F.col("r6") * 365).cast("int")).alias("transaction_date"),
        arr_ciudades[(F.col("r1") * 5).cast("int")].alias("store_city"),
        arr_pagos[(F.col("r2") * 4).cast("int")].alias("payment_method"),
        # Descuento: ~10% de nulos para limpiar en Silver
        F.when(F.col("r7") > 0.9, F.lit(None).cast("double"))
         .otherwise((F.col("r4") * 4).cast("int") * 0.05).alias("discount_pct"),
    )
)

print(f"Registros generados: {df_bronze_raw.count():,}")
print("\nEsquema:")
df_bronze_raw.printSchema()
df_bronze_raw.show(5, truncate=False)

Registros generados: 1,000

Esquema:
root
 |-- transaction_id: string (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_city: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- discount_pct: double (nullable = true)

+--------------+-----------+---------------+-----------+--------+----------+----------------+----------+--------------+-------------------+
|transaction_id|customer_id|product_name   |category   |quantity|unit_price|transaction_date|store_city|payment_method|discount_pct       |
+--------------+-----------+---------------+-----------+--------+----------+----------------+----------+--------------+-------------------+
|TXN-00001     |C0018      |Aceite Oliva   |Alimentos  |8       |1806.72   |2024-09-05      |Bogot

In [0]:
(
    df_bronze_raw
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DB_NAME}.bronze_ventas")   # ← cambio clave
)
print(f"Tabla Bronze guardada: {DB_NAME}.bronze_ventas")

Tabla Bronze guardada: taller2.bronze_ventas


In [0]:
# =============================================================
# 1.3 — EXPLORAR LA CAPA BRONZE
# Verificamos los datos antes de pasar a Silver
# =============================================================

df_bronze = spark.table(f"{DB_NAME}.bronze_ventas")   # ← en vez de read.load()
# El resto de la celda queda igual
print("=== Estadísticas básicas de Bronze ===")
print(f"Total de registros : {df_bronze.count():,}")
print(f"Columnas           : {len(df_bronze.columns)}")

print("\n=== Conteo de nulos por columna ===")
df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

print("=== Precios invalidos (unit_price < 0) ===")
print(f"Registros con precio invalido: {df_bronze.filter(F.col('unit_price') < 0).count()}")

print("\n=== Distribucion por categoria ===")
df_bronze.groupBy("category").count().orderBy("count", ascending=False).show()

print("=== Rango de fechas ===")
df_bronze.select(
    F.min("transaction_date").alias("fecha_min"),
    F.max("transaction_date").alias("fecha_max")
).show()

=== Estadísticas básicas de Bronze ===
Total de registros : 1,000
Columnas           : 10

=== Conteo de nulos por columna ===
+--------------+-----------+------------+--------+--------+----------+----------------+----------+--------------+------------+
|transaction_id|customer_id|product_name|category|quantity|unit_price|transaction_date|store_city|payment_method|discount_pct|
+--------------+-----------+------------+--------+--------+----------+----------------+----------+--------------+------------+
|             0|          0|           0|       0|       0|         0|               0|         0|             0|         109|
+--------------+-----------+------------+--------+--------+----------+----------------+----------+--------------+------------+

=== Precios invalidos (unit_price < 0) ===
Registros con precio invalido: 37

=== Distribucion por categoria ===
+-----------+-----+
|   category|count|
+-----------+-----+
|       Ropa|  225|
|Electronica|  203|
|  Alimentos|  191|
|   

---
## Parte 2 — Silver: Limpieza y transformación

La capa Silver aplica reglas de **calidad de datos** y **enriquecimiento**:

| Problema encontrado en Bronze | Acción en Silver |
|---|---|
| `unit_price` negativo (~3%) | Eliminar el registro |
| `discount_pct` nulo (~10%) | Rellenar con `0.0` (sin descuento) |
| Ausencia de columnas derivadas | Calcular `total_bruto`, `total_descuento`, `total_neto` |
| Sin dimensiones temporales | Extraer `year`, `month`, `month_name` |
| Sin auditoría | Agregar `processed_at` (timestamp de procesamiento) |

In [0]:
# =============================================================
# 2.1 — TRANSFORMACIÓN SILVER
# Leer Bronze → limpiar → enriquecer → guardar Silver
# =============================================================

from pyspark.sql import functions as F

df_bronze = spark.table(f"{DB_NAME}.bronze_ventas")   # ← cambio
# El resto de la celda (transformaciones) queda igual

df_silver = (
    df_bronze

    # --- LIMPIEZA ---
    # 1. Eliminar registros con precio inválido
    .filter(F.col("unit_price") > 0)
    # 2. Rellenar nulos en discount_pct con 0 (sin descuento)
    .fillna({"discount_pct": 0.0})

    # --- ENRIQUECIMIENTO ---
    # 3. Calcular totales de la transacción
    .withColumn("total_bruto",
        F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("total_descuento",
        F.round(F.col("total_bruto") * F.col("discount_pct"), 2))
    .withColumn("total_neto",
        F.round(F.col("total_bruto") - F.col("total_descuento"), 2))

    # 4. Extraer dimensiones de fecha
    .withColumn("year",       F.year("transaction_date"))
    .withColumn("month",      F.month("transaction_date"))
    .withColumn("month_name", F.date_format("transaction_date", "MMMM"))
    .withColumn("day_of_week",F.date_format("transaction_date", "EEEE"))

    # 5. Timestamp de procesamiento (auditoría)
    .withColumn("processed_at", F.current_timestamp())
)

registros_bronze = df_bronze.count()
registros_silver = df_silver.count()

print(f"Registros Bronze  : {registros_bronze:,}")
print(f"Registros Silver  : {registros_silver:,}")
print(f"Eliminados        : {registros_bronze - registros_silver:,} (precios invalidos)")

print("\nEsquema Silver:")
df_silver.printSchema()

Registros Bronze  : 1,000
Registros Silver  : 963
Eliminados        : 37 (precios invalidos)

Esquema Silver:
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_city: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- discount_pct: double (nullable = false)
 |-- total_bruto: double (nullable = true)
 |-- total_descuento: double (nullable = true)
 |-- total_neto: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- processed_at: timestamp (nullable = false)



In [0]:
# =============================================================
# 2.2 — GUARDAR Y VERIFICAR SILVER
# =============================================================

# Guardar como Delta Table, particionado por año y mes
# El particionado acelera consultas que filtran por fecha
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .saveAsTable(f"{DB_NAME}.silver_ventas")   # ← cambio
)
print(f"Tabla Silver guardada: {DB_NAME}.silver_ventas")

# Verificar nulos — leer con spark.table()
df_silver_check = spark.table(f"{DB_NAME}.silver_ventas")
# El resto de la celda queda igual

print("\n=== Verificacion de nulos en Silver (deben ser 0) ===")
df_silver_check.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["unit_price", "discount_pct", "total_neto"]
]).show()

print("=== Muestra de datos Silver ===")
display(df_silver_check.select(
    "transaction_id", "customer_id", "product_name", "category",
    "quantity", "unit_price", "discount_pct", "total_neto",
    "transaction_date", "year", "month"
).limit(10))

Tabla Silver guardada: taller2.silver_ventas

=== Verificacion de nulos en Silver (deben ser 0) ===
+----------+------------+----------+
|unit_price|discount_pct|total_neto|
+----------+------------+----------+
|         0|           0|         0|
+----------+------------+----------+

=== Muestra de datos Silver ===


transaction_id,customer_id,product_name,category,quantity,unit_price,discount_pct,total_neto,transaction_date,year,month
TXN-00269,C0121,Arroz Premium,Alimentos,3,1262.5,0.05,3598.12,2024-11-11,2024,11
TXN-00344,C0121,Escritorio Pie,Ropa,9,1953.24,0.15000000000000002,14942.29,2024-11-28,2024,11
TXN-00367,C0050,Silla Ergonomica,Muebles,1,1880.61,0.0,1880.61,2024-11-09,2024,11
TXN-00373,C0185,Zapatos Cuero,Alimentos,5,919.68,0.1,4138.56,2024-11-29,2024,11
TXN-00888,C0065,Zapatos Cuero,Libros,2,141.34,0.0,282.68,2024-11-29,2024,11
TXN-00890,C0065,Camiseta Algodon,Ropa,1,1918.24,0.0,1918.24,2024-11-08,2024,11
TXN-00907,C0125,Historia IA,Ropa,7,274.99,0.1,1732.44,2024-11-14,2024,11
TXN-00919,C0079,Historia IA,Libros,2,362.55,0.0,725.1,2024-11-12,2024,11
TXN-00939,C0117,Zapatos Cuero,Libros,5,1810.86,0.1,8148.87,2024-11-21,2024,11
TXN-00946,C0061,Silla Ergonomica,Ropa,2,1801.72,0.0,3603.44,2024-11-18,2024,11


---
## Parte 3 — Gold: Métricas y agregaciones

La capa Gold contiene tablas **pre-agregadas** listas para consumo analítico (dashboards, reportes, ML).  
Cada tabla Gold responde una pregunta de negocio específica.

Crearemos tres tablas Gold:
| Tabla | Pregunta de negocio |
|---|---|
| `ventas_categoria_mes` | ¿Cuáles categorías generan más ingresos por mes? |
| `top_clientes` | ¿Quiénes son los clientes de mayor valor? |
| `metodos_pago` | ¿Qué método de pago es más usado y cuál genera más descuento? |

In [0]:
# =============================================================
# 3.1 — GOLD: VENTAS POR CATEGORÍA Y MES
# =============================================================

from pyspark.sql import functions as F

df_silver = spark.table(f"{DB_NAME}.silver_ventas")

gold_cat_mes = (
    df_silver
    .groupBy("year", "month", "month_name", "category")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.sum("quantity").alias("unidades_vendidas"),
        F.round(F.sum("total_bruto"), 2).alias("ingresos_brutos"),
        F.round(F.sum("total_descuento"), 2).alias("total_descuentos"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("unit_price"), 2).alias("precio_promedio"),
    )
    .orderBy("year", "month", F.col("ingresos_netos").desc())
)

print("=== Ventas por categoria y mes (primeros 15 registros) ===")
display(gold_cat_mes.limit(15))

# Guardar como Delta
gold_cat_mes.write.format("delta").mode("overwrite").saveAsTable(f"{DB_NAME}.gold_cat_mes")


=== Ventas por categoria y mes (primeros 15 registros) ===


year,month,month_name,category,num_transacciones,unidades_vendidas,ingresos_brutos,total_descuentos,ingresos_netos,precio_promedio
2024,1,January,Alimentos,24,139,135138.29,14779.94,120358.35,990.24
2024,1,January,Ropa,24,142,129439.01,14310.62,115128.39,943.62
2024,1,January,Electronica,18,94,99877.05,5098.55,94778.5,1004.27
2024,1,January,Muebles,18,88,83826.77,6790.15,77036.62,1048.64
2024,1,January,Libros,13,79,75746.78,9134.99,66611.79,925.39
2024,2,February,Libros,18,98,113544.89,11578.3,101966.59,1208.24
2024,2,February,Ropa,19,93,92681.4,6775.8,85905.6,982.64
2024,2,February,Muebles,20,104,93070.22,7799.53,85270.69,853.97
2024,2,February,Electronica,14,76,81722.45,10277.03,71445.42,909.25
2024,2,February,Alimentos,10,58,68095.85,7745.72,60350.13,1145.97


In [0]:
# =============================================================
# 3.2 — GOLD: TOP CLIENTES POR GASTO TOTAL
# =============================================================

gold_top_clientes = (
    df_silver
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("num_compras"),
        F.round(F.sum("total_neto"), 2).alias("gasto_total"),
        F.round(F.avg("total_neto"), 2).alias("gasto_promedio"),
        F.round(F.sum("quantity"), 0).alias("unidades_compradas"),
        F.countDistinct("category").alias("categorias_distintas"),
    )
    .withColumn(
        "segmento",
        F.when(F.col("gasto_total") >= 5000, "VIP")
         .when(F.col("gasto_total") >= 2000, "Premium")
         .when(F.col("gasto_total") >= 500,  "Regular")
         .otherwise("Ocasional")
    )
    .orderBy(F.col("gasto_total").desc())
)

print("=== Top 15 clientes por gasto total ===")
display(gold_top_clientes.limit(15))

print("\n=== Distribucion de clientes por segmento ===")
gold_top_clientes.groupBy("segmento").count().orderBy("count", ascending=False).show()

# Guardar como Delta
gold_top_clientes.write.format("delta").mode("overwrite").saveAsTable(f"{DB_NAME}.gold_top_clientes")

=== Top 15 clientes por gasto total ===


customer_id,num_compras,gasto_total,gasto_promedio,unidades_compradas,categorias_distintas,segmento
C0156,12,65726.5,5477.21,57,4,VIP
C0152,9,62928.25,6992.03,48,5,VIP
C0174,7,55302.58,7900.37,43,5,VIP
C0198,9,54444.66,6049.41,42,4,VIP
C0006,9,54443.03,6049.23,54,4,VIP
C0089,8,53632.83,6704.1,45,4,VIP
C0179,6,53362.75,8893.79,51,4,VIP
C0024,7,52604.77,7514.97,44,5,VIP
C0165,9,50770.89,5641.21,49,5,VIP
C0121,6,50353.73,8392.29,41,4,VIP



=== Distribucion de clientes por segmento ===
+---------+-----+
| segmento|count|
+---------+-----+
|      VIP|  183|
|  Premium|   10|
|  Regular|    5|
|Ocasional|    1|
+---------+-----+



In [0]:
# =============================================================
# 3.3 — GOLD: ANÁLISIS POR MÉTODO DE PAGO
# También usamos Spark SQL como alternativa a la API de DataFrames
# =============================================================

# Registrar la tabla Silver como vista temporal para usar SQL
df_silver = spark.table(f"{DB_NAME}.silver_ventas")
df_silver.createOrReplaceTempView("silver_ventas")


gold_pagos = spark.sql("""
    SELECT
        payment_method,
        COUNT(*)                                         AS num_transacciones,
        ROUND(SUM(total_neto), 2)                        AS ingresos_netos,
        ROUND(AVG(total_neto), 2)                        AS ticket_promedio,
        ROUND(AVG(discount_pct) * 100, 1)               AS descuento_promedio_pct,
        ROUND(SUM(total_descuento), 2)                   AS total_descuentos_otorgados
    FROM silver_ventas
    GROUP BY payment_method
    ORDER BY ingresos_netos DESC
""")

print("=== Analisis por metodo de pago ===")
display(gold_pagos)

# Guardar como Delta
gold_pagos.write.format("delta").mode("overwrite").saveAsTable(f"{DB_NAME}.gold_metodos_pago")

=== Analisis por metodo de pago ===


payment_method,num_transacciones,ingresos_netos,ticket_promedio,descuento_promedio_pct,total_descuentos_otorgados
transferencia,246,1171856.1,4763.64,6.6,119829.34
efectivo,240,1133692.05,4723.72,7.0,120598.97
tarjeta_credito,241,1072886.31,4451.81,5.7,100736.81
tarjeta_debito,236,1071449.96,4540.04,7.0,110756.3


---
## Parte 4 — Delta Lake: Capacidades avanzadas

Delta Lake es un **formato de tabla abierto** construido sobre Parquet que agrega:

| Capacidad | Qué significa |
|---|---|
| **ACID Transactions** | Las escrituras son atómicas (todo o nada) |
| **Transaction Log** | Cada operación queda registrada en `_delta_log/` |
| **Time Travel** | Puedes leer versiones anteriores de la tabla |
| **Schema Enforcement** | Rechaza escrituras que no cumplan el esquema |
| **OPTIMIZE / ZORDER** | Compacta archivos pequeños y mejora performance |

En esta sección exploraremos estas capacidades sobre la tabla Silver.

In [0]:
# =============================================================
# 4.1 — TRANSACTION LOG: DESCRIBE HISTORY
# Cada escritura crea una nueva versión registrada en el log
# =============================================================



print("=== Historial de la tabla Silver ===")
display(
    spark.sql(f"DESCRIBE HISTORY {DB_NAME}.silver_ventas")
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
)

=== Historial de la tabla Silver ===


version,timestamp,operation,operationParameters,operationMetrics
0,2026-04-19T20:45:42.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""year"",""month""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)","Map(numFiles -> 12, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 963, numOutputBytes -> 93854)"


In [0]:
# =============================================================
# 4.2 — SIMULAR UNA ACTUALIZACIÓN PARA CREAR UNA NUEVA VERSIÓN
# Escribimos datos adicionales (append) para generar la versión 1
# =============================================================

# Tomar 50 registros adicionales de Silver y agregarlos (simula un lote nuevo)
df_nuevos = df_silver.limit(50).withColumn("processed_at", F.current_timestamp())

df_nuevos.write.format("delta").mode("append").saveAsTable(f"{DB_NAME}.silver_ventas")
display(spark.sql(f"DESCRIBE HISTORY {DB_NAME}.silver_ventas").select("version","timestamp","operation"))

version,timestamp,operation
1,2026-04-19T20:46:04.000Z,WRITE
0,2026-04-19T20:45:42.000Z,CREATE OR REPLACE TABLE AS SELECT


In [0]:
# =============================================================
# 4.3 — TIME TRAVEL: LEER UNA VERSIÓN ANTERIOR
# Esto es posible gracias al transaction log de Delta Lake
# =============================================================

# Leer la versión actual (más reciente)
df_version_actual = spark.table(f"{DB_NAME}.silver_ventas")
df_version_0 = spark.read.format("delta").option("versionAsOf", 0).table(f"{DB_NAME}.silver_ventas")
print(f"Registros actuales : {df_version_actual.count():,}")
print(f"Registros v0       : {df_version_0.count():,}")

# También se puede hacer time travel por timestamp:
# spark.read.format("delta").option("timestampAsOf", "2024-01-01").load(...)

Registros actuales : 1,013
Registros v0       : 963


In [0]:
# =============================================================
# 4.4 — OPTIMIZE: COMPACTAR ARCHIVOS PEQUEÑOS
# En una tabla real con muchos appends se generan archivos pequeños.
# OPTIMIZE los fusiona en archivos más grandes (mejor performance).
# =============================================================

# OPTIMIZE compacta los archivos Parquet de la tabla
spark.sql(f"OPTIMIZE {DB_NAME}.silver_ventas")
print("OPTIMIZE completado.")

# Verificar el historial después del OPTIMIZE
display(spark.sql(f"DESCRIBE HISTORY {DB_NAME}.silver_ventas").limit(5))


OPTIMIZE completado.


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-04-19T20:46:11.000Z,73368588888216,jeronimo.velasquez.escobar@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2168594438775731),d4dd6a57-0b65-494f-bc09-dae7af5234d2,0419-202454-8144sc0b-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 14641, p25FileSize -> 8444, numDeletionVectorsRemoved -> 0, minFileSize -> 8444, numAddedFiles -> 1, maxFileSize -> 8444, p75FileSize -> 8444, p50FileSize -> 8444, numAddedBytes -> 8444)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-19T20:46:04.000Z,73368588888216,jeronimo.velasquez.escobar@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2168594438775731),d8b875de-a48c-4b6a-8528-f939f6595471,0419-202454-8144sc0b-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 50, numOutputBytes -> 7043)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-19T20:45:42.000Z,73368588888216,jeronimo.velasquez.escobar@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""year"",""month""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2168594438775731),714f996f-81b2-48e9-b64d-3c2d9bc566b2,0419-202454-8144sc0b-v2n,null,WriteSerializable,false,"Map(numFiles -> 12, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 963, numOutputBytes -> 93854)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
# =============================================================
# 4.5 — CONSULTAS SQL SOBRE TABLAS DELTA
# Delta Lake soporta SQL completo, incluyendo DML (UPDATE, DELETE, MERGE)
# =============================================================

# Registrar tablas Gold como vistas temporales
spark.sql(f"SELECT * FROM {DB_NAME}.gold_cat_mes").createOrReplaceTempView("gold_cat_mes")
spark.sql(f"SELECT * FROM {DB_NAME}.gold_top_clientes").createOrReplaceTempView("gold_clientes")

# Consulta 1: Top 3 categorías por ingresos netos totales
print("=== Top 3 categorias por ingresos totales ===")
spark.sql("""
    SELECT category,
           SUM(num_transacciones)  AS transacciones,
           SUM(ingresos_netos)     AS ingresos_netos_total
    FROM gold_cat_mes
    GROUP BY category
    ORDER BY ingresos_netos_total DESC
    LIMIT 3
""").show()

# Consulta 2: Distribución de segmentos de clientes
print("=== Distribucion por segmento ===")
spark.sql("""
    SELECT segmento,
           COUNT(*)                    AS num_clientes,
           ROUND(AVG(gasto_total), 2)  AS gasto_promedio
    FROM gold_clientes
    GROUP BY segmento
    ORDER BY gasto_promedio DESC
""").show()

=== Top 3 categorias por ingresos totales ===
+--------+-------------+--------------------+
|category|transacciones|ingresos_netos_total|
+--------+-------------+--------------------+
|    Ropa|          216|  1020354.1499999999|
|  Libros|          185|            885481.5|
| Muebles|          189|   859351.6699999999|
+--------+-------------+--------------------+

=== Distribucion por segmento ===
+---------+------------+--------------+
| segmento|num_clientes|gasto_promedio|
+---------+------------+--------------+
|      VIP|         183|      24062.29|
|  Premium|          10|       3901.69|
|  Regular|           5|       1447.57|
|Ocasional|           1|        230.46|
+---------+------------+--------------+



---
## Parte 5 — Ejercicios integradores

Completa los siguientes ejercicios usando los datos de las capas Silver y Gold.  
Puedes usar tanto la API de DataFrames como Spark SQL.

> **Tip:** recuerda que `df_silver` ya está cargado. También puedes usar `spark.read.format("delta").load(...)` para releer cualquier capa.

**E1** — ¿Cuál es el producto más vendido (en unidades totales) en cada categoría? Muestra: categoría, producto y total de unidades.

In [0]:
# =============================================================
# E1 — Producto más vendido (en unidades) por categoría
# =============================================================
from pyspark.sql import Window

df_silver = spark.table(f"{DB_NAME}.silver_ventas")

w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())

(
    df_silver
    .groupBy("category", "product_name")
    .agg(F.sum("quantity").alias("total_unidades"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy("category")
    .show()
)

+-----------+--------------+--------------+
|   category|  product_name|total_unidades|
+-----------+--------------+--------------+
|  Alimentos|  Jean Clasico|            91|
|Electronica|  Leche Entera|           105|
|     Libros|  Aceite Oliva|           104|
|    Muebles|  Jean Clasico|           103|
|       Ropa|Escritorio Pie|           156|
+-----------+--------------+--------------+



**E2** — Calcula la **tasa de descuento promedio** y los **ingresos netos por ciudad** (`store_city`). Ordena de mayor a menor ingreso.

In [0]:
# =============================================================
# E2 — Ingresos netos y descuento promedio por ciudad
# =============================================================

(
    df_silver
    .groupBy("store_city")
    .agg(
        F.round(F.avg("discount_pct") * 100, 1).alias("descuento_promedio_pct"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.count("transaction_id").alias("num_transacciones")
    )
    .orderBy(F.col("ingresos_netos").desc())
    .show()
)

+------------+----------------------+--------------+-----------------+
|  store_city|descuento_promedio_pct|ingresos_netos|num_transacciones|
+------------+----------------------+--------------+-----------------+
|        Cali|                   6.3|     971664.48|              211|
|   Cartagena|                   7.1|     955214.79|              202|
|      Bogota|                   6.6|     934133.18|              196|
|    Medellin|                   6.4|     914252.61|              210|
|Barranquilla|                   6.4|     908426.14|              194|
+------------+----------------------+--------------+-----------------+



**E3** — Construye una nueva tabla Gold llamada `gold_tendencia_semanal` que muestre por semana del año: número de transacciones, ingresos netos y ticket promedio. Guárdala en Delta en `{GOLD_PATH}/tendencia_semanal/`.

> Pista: usa `F.weekofyear("transaction_date")` para extraer la semana.

In [0]:
# =============================================================
# E3 — Gold: tendencia semanal
# =============================================================

gold_semanal = (
    df_silver
    .withColumn("week", F.weekofyear("transaction_date"))
    .groupBy("year", "week")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
    )
    .orderBy("year", "week")
)

gold_semanal.write.format("delta").mode("overwrite").saveAsTable(f"{DB_NAME}.gold_tendencia_semanal")

print(f"Tabla guardada: {DB_NAME}.gold_tendencia_semanal")
display(gold_semanal)

Tabla guardada: taller2.gold_tendencia_semanal


year,week,num_transacciones,ingresos_netos,ticket_promedio
2024,1,24,153529.25,6397.05
2024,2,24,94996.29,3958.18
2024,3,18,89394.69,4966.37
2024,4,27,119868.46,4439.57
2024,5,15,105330.79,7022.05
2024,6,17,82561.72,4856.57
2024,7,19,87561.58,4608.5
2024,8,24,116061.69,4835.9
2024,9,23,76969.12,3346.48
2024,10,17,65395.17,3846.77


**E4 (Avanzado)** — Usa el **time travel de Delta Lake** para responder: ¿cuánto cambió el total de ingresos netos entre la versión 0 y la versión actual de la tabla Silver? Muestra ambos valores y la diferencia.

In [0]:
# =============================================================
# E4 — Time Travel: diferencia de ingresos entre v0 y actual
# =============================================================

ingresos_v0 = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .table(f"{DB_NAME}.silver_ventas")
    .agg(F.round(F.sum("total_neto"), 2).alias("ingresos_v0"))
    .collect()[0]["ingresos_v0"]
)

ingresos_actual = (
    spark.table(f"{DB_NAME}.silver_ventas")
    .agg(F.round(F.sum("total_neto"), 2).alias("ingresos_actual"))
    .collect()[0]["ingresos_actual"]
)

print(f"Ingresos v0      : ${ingresos_v0:>12,.2f}")
print(f"Ingresos actual  : ${ingresos_actual:>12,.2f}")
print(f"Diferencia       : ${ingresos_actual - ingresos_v0:>12,.2f}")

Ingresos v0      : $4,449,884.42
Ingresos actual  : $4,683,691.20
Diferencia       : $  233,806.78


---
## Soluciones de referencia

Descomenta cada bloque para ver la solución.

In [0]:
# --- E1 ---
# from pyspark.sql import Window
# w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())
#
# (
#     df_silver
#     .groupBy("category", "product_name")
#     .agg(F.sum("quantity").alias("total_unidades"))
#     .withColumn("rank", F.rank().over(w))
#     .filter(F.col("rank") == 1)
#     .drop("rank")
#     .orderBy("category")
#     .show()
# )

In [0]:
# --- E2 ---
# (
#     df_silver
#     .groupBy("store_city")
#     .agg(
#         F.round(F.avg("discount_pct") * 100, 1).alias("descuento_promedio_pct"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.count("transaction_id").alias("num_transacciones")
#     )
#     .orderBy(F.col("ingresos_netos").desc())
#     .show()
# )

In [0]:
# --- E3 ---
# gold_semanal = (
#     df_silver
#     .withColumn("week", F.weekofyear("transaction_date"))
#     .groupBy("year", "week")
#     .agg(
#         F.count("transaction_id").alias("num_transacciones"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
#     )
#     .orderBy("year", "week")
# )
# gold_semanal.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/tendencia_semanal/")
# display(gold_semanal)

In [0]:
# --- E4 ---
# ingresos_v0 = (
#     spark.read.format("delta").option("versionAsOf", 0).load(f"{SILVER_PATH}/ventas/")
#     .agg(F.round(F.sum("total_neto"), 2).alias("ingresos_v0"))
#     .collect()[0]["ingresos_v0"]
# )
# ingresos_actual = (
#     spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")
#     .agg(F.round(F.sum("total_neto"), 2).alias("ingresos_actual"))
#     .collect()[0]["ingresos_actual"]
# )
# print(f"Ingresos v0      : ${ingresos_v0:>12,.2f}")
# print(f"Ingresos actual  : ${ingresos_actual:>12,.2f}")
# print(f"Diferencia       : ${ingresos_actual - ingresos_v0:>12,.2f}")

---
## Limpieza (opcional)

Ejecuta esta celda al finalizar si quieres liberar el espacio usado en DBFS.

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {DB_NAME} CASCADE")
print(f"Schema '{DB_NAME}' eliminado correctamente.")

Schema 'taller2' eliminado correctamente.
